# WISER Module 5 — Data Visualization and Communication

This notebook supports **Rise Sections 5.3–5.4**. Your product is **one chart and one-page PDF decision brief**, along with your notebook, a ~200-word tradeoff reflection, and documented outputs.

1. Select **File → Save a copy in Drive** before making changes.
2. Choose **Foundational** or **Applied** below, as you did in Rise. Complete only your chosen pathway.
3. Run the notebook in order. Read each result, replace the bracketed learner responses, and rerun the export at the end. **Run all alone does not complete the assignment.**
4. Before submitting, set your Colab copy to **view access** for your instructor, download the one-page PDF, and download the outputs ZIP.

**Privacy:** Use only provided synthetic data or approved public, aggregate data. Do not upload PHI or identifiable records.

**Data distinction:** The supplied `post_overdose_data.csv` is a synthetic *event-level* file. It does not contain a death count or population denominators. This notebook charts recorded synthetic events. It does **not** calculate overdose mortality or a population rate. Rise's illustrative county-versus-state mortality scenario is separate from what this specific CSV can establish.

In [ ]:
PATHWAY = "Foundational"  # Change to "Applied" only if you chose 5.32 in Rise.
assert PATHWAY in ("Foundational", "Applied")
print("Your pathway:", PATHWAY)

## 1. Load and examine your data (Rise 5.4, step 1)

**Foundational:** The notebook loads the provided synthetic CSV automatically.

**Applied:** Set `APPLIED_SOURCE` to the path of an approved public aggregate CSV (such as a properly exported CDC WONDER table). You may explicitly use the synthetic file for offline development by setting `APPLIED_SOURCE = "synthetic"`. Applied learners must document the source, reporting period, outcome definition, geography, and any filter or ICD code choices. The rest of this notebook demonstrates the Foundational workflow; an Applied import must be mapped to the analysis fields before proceeding.

In [ ]:
from pathlib import Path
import json, re, shutil, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
try:
    from IPython.display import display
except ImportError:
    display = print

DATA_URL = "https://raw.githubusercontent.com/mcbacloudlab/WISERTraining-Modules-/main/Module%205/Data/post_overdose_data.csv"
OUT = Path("module5_outputs")
OUT.mkdir(exist_ok=True)
APPLIED_SOURCE = ""  # Applied: local path to an approved public aggregate CSV, or "synthetic" for offline development.
if PATHWAY == "Foundational" or APPLIED_SOURCE == "synthetic":
    events = pd.read_csv(DATA_URL)
    source_name = "WISER synthetic post_overdose_data.csv"
    source_url = DATA_URL
    synthetic_used = True
elif APPLIED_SOURCE:
    events = pd.read_csv(APPLIED_SOURCE)
    source_name = APPLIED_SOURCE
    source_url = APPLIED_SOURCE
    synthetic_used = False
else:
    raise ValueError("Applied pathway: set APPLIED_SOURCE to a public aggregate CSV path, or 'synthetic' for offline development.")
print("Source:", source_name, "| rows:", len(events), "| columns:", len(events.columns))
display(events.head(5))
display(pd.DataFrame({"field": events.columns, "missing": events.isna().sum().to_numpy()}))

### Read the dataset before charting

The provided file's `event_id` identifies an event, `event_date` gives its date, `opioid_involved` marks whether opioids were involved, and `provisional_flag` identifies records subject to revision. A row is **not necessarily a death**. A count of rows is **not** a population rate. Also inspect whether duplicate IDs or inconsistent dates need cleaning.

In [ ]:
if synthetic_used:
    required = {"event_id", "event_date", "opioid_involved", "provisional_flag", "county", "svi_quartile"}
    missing = required - set(events.columns)
    if missing: raise ValueError(f"Required fields missing: {sorted(missing)}")
    events["event_date"] = pd.to_datetime(events["event_date"], errors="coerce")
    print("Duplicate event IDs:", events["event_id"].duplicated().sum())
    print("Unparseable dates:", events["event_date"].isna().sum())
    print("Period:", events["event_date"].min().date(), "to", events["event_date"].max().date())
    print("Provisional rows by year:")
    display(pd.crosstab(events["event_date"].dt.year, events["provisional_flag"]))
    if events["event_id"].duplicated().any() or events["event_date"].isna().any():
        raise ValueError("Review duplicate IDs or dates before plotting.")
else:
    print("APPLIED TASK: document source structure, duplicates, missingness, inclusion criteria, and cleaning in your own cells.")

## 2. Record definitions and CDE alignment (Rise 5.4, step 2)

The table below **documents variables and possible HEAL CDE topics**. It does not assign a HEAL CDE identifier without a verified definition. For an Applied submission, complete the target identifier/reference fields using an authoritative CDE source; mark `unverified` or `no suitable mapping` when appropriate. Do not infer a population denominator from an event count.

In [ ]:
cde_crosswalk = pd.DataFrame([
    {"source_field":"event_date", "source_definition":"Date of synthetic event", "candidate_CDE_topic":"time", "verified_target_id":"", "mapping_status":"unverified", "evidence":""},
    {"source_field":"county", "source_definition":"County assigned to synthetic event", "candidate_CDE_topic":"geography", "verified_target_id":"", "mapping_status":"unverified", "evidence":""},
    {"source_field":"substance_involved_primary", "source_definition":"Primary substance label, not necessarily all substances involved", "candidate_CDE_topic":"drug class", "verified_target_id":"", "mapping_status":"unverified", "evidence":""},
    {"source_field":"age_group", "source_definition":"Age category", "candidate_CDE_topic":"demographics", "verified_target_id":"", "mapping_status":"unverified", "evidence":""},
    {"source_field":"provisional_flag", "source_definition":"Event record is flagged provisional", "candidate_CDE_topic":"data status", "verified_target_id":"", "mapping_status":"unverified", "evidence":""},
])
if PATHWAY == "Applied" and not synthetic_used:
    # Replace with documented fields from your actual public source.
    cde_crosswalk = pd.DataFrame([{col: "" for col in cde_crosswalk.columns}])
    print("APPLIED: populate with your source fields and verified HEAL CDE references.")
display(cde_crosswalk)

## 3. Clean, count, and assess rate readiness (Rise 5.4, steps 3–4)

The Foundational analysis uses **monthly counts of recorded opioid-involved synthetic events**. A rate per 100,000 would require a matching population denominator for the same place and period. A mortality rate would additionally require a verified death outcome. Those inputs are absent here. The notebook checks and says so instead of inventing them.

**Applied:** Build and document your own ingestion and cleaning pipeline for a public source. If you use CDC WONDER, state your selected causes, multiple-cause drug codes, years, geography, suppression rules, and population denominator before calculating mortality rates. If you use this synthetic file for development, keep the event-count label.

In [ ]:
if synthetic_used:
    analysis = events.loc[events["opioid_involved"].eq(1)].copy()
    analysis["month"] = analysis["event_date"].dt.to_period("M").dt.to_timestamp()
    monthly = (analysis.groupby("month", as_index=False)
              .agg(recorded_events=("event_id", "size"), provisional_events=("provisional_flag", "sum")))
    all_months = pd.date_range(events["event_date"].min().to_period("M").to_timestamp(),
                               events["event_date"].max().to_period("M").to_timestamp(), freq="MS")
    monthly = pd.DataFrame({"month":all_months}).merge(monthly, on="month", how="left")
    monthly[["recorded_events", "provisional_events"]] = monthly[["recorded_events", "provisional_events"]].fillna(0).astype(int)
    # A month is marked provisional when at least one of its source records is flagged.
    monthly["provisional"] = monthly["provisional_events"].gt(0)
    print("Outcome: count of recorded opioid-involved synthetic events per month")
    print("Population rate available? No; matching population denominator is absent.")
    print("Overdose mortality rate available? No; no verified fatal outcome or denominator.")
    display(monthly.tail(15))
else:
    # Applied learners implement and document the transformation from their public
    # aggregate source to a plot-ready table; this accepted schema is an example.
    needed = {"month", "recorded_events", "provisional"}
    if not needed.issubset(events.columns):
        raise ValueError("APPLIED: add a cleaning cell that creates month, recorded_events, and provisional columns; document any population rate separately. Found columns: " + str(list(events.columns)))
    monthly = events[["month", "recorded_events", "provisional"]].copy()
    monthly["month"] = pd.to_datetime(monthly["month"], errors="coerce")
    monthly["recorded_events"] = pd.to_numeric(monthly["recorded_events"], errors="coerce")
    if monthly["month"].isna().any() or monthly["recorded_events"].isna().any():
        raise ValueError("APPLIED: correct missing/non-numeric dates or counts in your public table.")
    if monthly["month"].duplicated().any():
        raise ValueError("APPLIED: aggregate to exactly one row per month and document that aggregation.")
    monthly["provisional"] = monthly["provisional"].astype(bool)
    monthly = monthly.sort_values("month").reset_index(drop=True)
    print("APPLIED: verify that labels, measure, periods, provenance, and CDEs below describe your own data.")

## 4. Show uncertainty and consider equity (Rise 5.4, steps 5–6)

The chart will mark months based on the **actual** `provisional_flag`, rather than guessing that the last six months are provisional. This flag identifies records subject to revision; it does not tell us the size or direction of a future revision. A month with zero events may also reflect reporting coverage, so avoid claiming the population had no events.

The file includes `svi_quartile`, but it lacks tract-level denominators and a documented SVI merge. Examine it for context; do not rank communities or infer risk from raw counts. Write one equity limitation.

In [ ]:
if synthetic_used:
    display(events.groupby("svi_quartile", as_index=False).agg(recorded_events=("event_id", "size"), opioid_involved=("opioid_involved", "sum")))
    print("This is a descriptive count in a synthetic sample, not a community rate.")
equity_note = """[Describe one limitation of comparing these event counts by SVI or county without a population denominator or coverage information.]"""
print(equity_note)

## 5. Plan an accessible chart (Rise 5.1 and 5.4, step 7)

**Foundational chart:** A monthly line chart fits a question about change over time. The code prepares the data and draws the figure. **Your edits:** Write an accurate question, an audience, an action the audience could consider, a chart title, a caption that names the unit and period, and alt text that states what the pattern shows. Avoid an unsupported claim that mortality increased locally faster than statewide. Use the output above to decide what the data actually say.

**Visual checks:** Label counts explicitly, shade provisional months, use a direct note and a high-contrast line, and avoid using color alone to indicate data status. A chart title must reflect recorded synthetic events.

In [ ]:
audience = "County health department planning team"  # EDIT: choose your decision-maker.
question = "How did recorded opioid-involved synthetic event counts vary by month?"  # EDIT for your chosen audience.
action_goal = "Review outreach capacity and data quality before allocating resources."  # EDIT to a specific defensible action.
chart_title = "Recorded opioid-involved events by month"  # EDIT: say what is measured.
chart_caption = "Synthetic WISER events, Jan 2019–Mar 2025. Monthly counts; shaded months contain provisional records. Not a mortality or population rate."  # EDIT.
alt_text = """[Describe the high/low pattern or change visible in the chart, including the provisional period. Do not just write 'line chart'.]"""
print("Audience:", audience)
print("Question:", question)
print("Decision:", action_goal)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(monthly["month"], monthly["recorded_events"], color="#12647A", linewidth=2.3, marker="o", markersize=2.3)
# Shade each contiguous run of months containing provisional records.
for row in monthly.itertuples():
    if row.provisional:
        ax.axvspan(row.month, row.month + pd.DateOffset(months=1), color="#F1C04A", alpha=.34)
ax.set(title=chart_title, xlabel="Month", ylabel="Recorded events (count)")
ax.set_ylim(bottom=0)
ax.grid(axis="y", color="#DFE5E8")
ax.spines[["top", "right"]].set_visible(False)
ax.text(.99, .96, "Gold shading: provisional records", transform=ax.transAxes,
        ha="right", va="top", fontsize=9, bbox={"facecolor":"white","alpha":.9,"edgecolor":"none"})
fig.autofmt_xdate(rotation=0)
fig.tight_layout()
chart_path = OUT / "module5_primary_chart.png"
fig.savefig(chart_path, dpi=180, bbox_inches="tight")
plt.show()
print("Chart saved:", chart_path)
print("Alt text:", alt_text)

### Chart and language audit

Review every visible label and caption against the data: Does it say *recorded events*? Does the shading match the `provisional_flag`? Are units, dates, source, and limitations clear? Avoid stigmatizing labels. The simple word check below prompts human review; it cannot decide whether a word is appropriate in context.

In [ ]:
TERMS_TO_REVIEW = ["addict", "abuser", "drug abuse", "substance abuse", "junkie", "dirty", "clean"]
def wording_flags(text):
    return [term for term in TERMS_TO_REVIEW if re.search(r"(?<!\w)"+re.escape(term)+r"(?!\w)", str(text), re.I)]
for label, value in {"title":chart_title, "caption":chart_caption, "alt text":alt_text, "question":question}.items():
    print(label, "terms to review:", wording_flags(value))
accessibility_check = """[Confirm the unit, date range, source/caption, provisional cue, readable labels, alt text, and person-first wording. Note one change you made.]"""
print(accessibility_check)

## 6. Draft the one-page decision brief (Rise 5.1 and 5.4, step 8)

Complete five blocks: **question, plain-language finding, chart, caveats, action request**. Write a finding supported by this chart. Give **2–3 concrete caveats**, including that data are synthetic and include provisional records, and that event counts cannot establish a mortality rate. State a realistic action with an owner and timeframe. Do not invent a funding amount, source population, or county–state comparator. The cell below turns your completed text and chart into a one-page PDF.

In [ ]:
headline = "[Write one plain-language finding from the actual chart, with a number if you can support it.]"
caveats = [
    "[Describe the provisional months and how they limit interpretation.]",
    "[Explain that these are synthetic recorded events, not mortality rates or population rates.]",
    "[Optional: note reporting coverage, missing denominator, or equity limitation.]",
]
recommendation = "[State a specific, proportionate planning or data-quality action, who should take it, and when.]"
source_note = "WISER synthetic post_overdose_data.csv; selected opioid-involved records. Data through March 2025."
print("Question:", question, "\nFinding:", headline, "\nCaveats:", *caveats, sep="\n")
print("Action:", recommendation)

In [ ]:
# Keep the brief within one readable page. Shorten fields rather than allowing overlap.
limits = {"audience":100, "question":170, "headline":170, "caption":190,
          "recommendation":170, **{f"caveat {i}":110 for i in range(1,4)}}
values = {"audience":audience, "question":question, "headline":headline,
          "caption":chart_caption, "recommendation":recommendation,
          **{f"caveat {i}":c for i,c in enumerate(caveats,1)}}
long = [k for k,v in values.items() if len(v)>limits[k]]
if long: raise ValueError("Shorten these fields to fit the one-page brief: " + ", ".join(long))

brief_fig = plt.figure(figsize=(8.5, 11), facecolor="white")
brief_fig.text(.075, .966, "WISER Module 5 | Decision brief", fontsize=14, weight="bold", color="#143C4A")
brief_fig.text(.075, .935, f"Audience: {audience}", fontsize=9)
brief_fig.text(.075, .905, "Question", fontsize=10, weight="bold")
brief_fig.text(.075, .884, "\n".join(textwrap.wrap(question, 95)), fontsize=9.2, va="top")
brief_fig.text(.075, .825, "Key finding", fontsize=10, weight="bold")
brief_fig.text(.075, .804, "\n".join(textwrap.wrap(headline, 95)), fontsize=9.2, va="top")
chart_ax = brief_fig.add_axes([.075, .405, .85, .34])
chart_ax.imshow(plt.imread(chart_path))
chart_ax.axis("off")
brief_fig.text(.075, .394, "Figure. " + "\n".join(textwrap.wrap(chart_caption, 105)), fontsize=8.3, va="top")
brief_fig.text(.075, .33, "Key caveats", fontsize=10, weight="bold")
y = .306
for i, caveat in enumerate(caveats, 1):
    if caveat.strip():
        lines = textwrap.wrap(f"{i}. {caveat}", 112)
        brief_fig.text(.075, y, "\n".join(lines), fontsize=8.8, va="top", linespacing=1.2)
        y -= .018*len(lines) + .012
if y < .145: raise ValueError("Shorten the caveats to keep the brief on one readable page.")
brief_fig.text(.075, .141, "Recommended action", fontsize=10, weight="bold")
brief_fig.text(.075, .119, "\n".join(textwrap.wrap(recommendation, 105)), fontsize=8.8, va="top")
brief_fig.text(.075, .042, "Source: " + source_note, fontsize=7.5)
brief_fig.text(.075, .025, "Learning exercise | Synthetic event records; no population or mortality estimate" if synthetic_used else "Public data | See notebook and metadata for methods", fontsize=7.5)
brief_path = OUT / "Module_5_one_page_decision_brief.pdf"
with PdfPages(brief_path) as pdf:
    pdf.savefig(brief_fig)
plt.close(brief_fig)
print("Saved one-page PDF:", brief_path)
print("Review the PDF visually before submitting; edit text and rerun this cell if anything overlaps.")

## 7. Applied pathway extension (Rise 5.32)

Applied learners use public data and document their own ingestion, cleaning, denominator, suppression handling, and chart code. Rise requests a **HEAL CDE crosswalk table** alongside the chart, brief, and notebook. Edit the crosswalk above after verifying source definitions and identifiers. An optional SUDORS or SVI merge needs compatible geography, period, definitions, and access terms. Do not infer a rate from the synthetic event file.

If your Applied analysis uses a different table, replace the Foundational `monthly` build and document the new fields. Update every source line, title, caveat, and metadata field. A synthetic fallback is acceptable for offline practice only when you disclose it.

In [ ]:
applied_methods = """[Applied: describe source URL/release, public-use status, outcome and drug codes, geography, denominator, suppression, cleaning, and revisions. Add your implementation code cells above.]"""
if PATHWAY == "Applied":
    print("APPLIED CHECK: a public-data pipeline and verified CDE mapping are required; this text is only a prompt.")
    print(applied_methods)

## 8. Reflect, check, and export (Rise 5.3–5.4, step 9)

Write **about 200 words** on one tradeoff (for example: count versus rate, including provisional months, or omitting a county comparison). Explain what you chose, why, and what information would change your choice. Run the last cell after finishing the chart, brief, and responses.

**Submit:** (1) Colab notebook link with view access; (2) one-page PDF brief; (3) outputs folder/ZIP. Your reflection is saved in the notebook and outputs. For Applied, include the verified HEAL CDE crosswalk. Open the PDF and files before you submit.

In [ ]:
tradeoff_reflection = """[Replace with your ~200-word reflection on one evidence or communication tradeoff.]"""
print("Reflection words:", len(tradeoff_reflection.split()))

In [ ]:
required_text = {"alt text":alt_text, "accessibility check":accessibility_check,
                 "finding":headline, "action":recommendation, "tradeoff reflection":tradeoff_reflection,
                 "equity note":equity_note}
for i, caveat in enumerate(caveats, 1): required_text[f"caveat {i}"] = caveat
unfinished = [key for key,value in required_text.items() if "[" in str(value) and "]" in str(value)]
if unfinished: print("TO COMPLETE before submission:", ", ".join(unfinished))
if not 170 <= len(tradeoff_reflection.split()) <= 230:
    print("TO COMPLETE: write a roughly 200-word tradeoff reflection.")
if PATHWAY == "Applied" and (synthetic_used or "[" in applied_methods or cde_crosswalk["mapping_status"].eq("unverified").all()):
    print("APPLIED CHECK: complete your public-data pipeline and documented CDE crosswalk; disclose any synthetic fallback.")

monthly.to_csv(OUT / "monthly_recorded_events.csv", index=False)
cde_crosswalk.to_csv(OUT / "cde_alignment_table.csv", index=False)
(OUT / "reflection.txt").write_text(tradeoff_reflection.strip()+"\n", encoding="utf-8")
(OUT / "chart_alt_text.txt").write_text(alt_text.strip()+"\n", encoding="utf-8")
(OUT / "brief_text.md").write_text(
    f"# Decision brief\n\nAudience: {audience}\n\nQuestion: {question}\n\n"
    f"Finding: {headline}\n\nCaveats:\n"+"\n".join(f"- {x}" for x in caveats)
    +f"\n\nRecommended action: {recommendation}\n\nSource: {source_note}\n", encoding="utf-8")
metadata = {"module":"WISER Module 5", "pathway":PATHWAY, "source":source_name,
            "source_url_or_path":source_url, "rows":len(events), "synthetic_used":synthetic_used,
            "measure":"monthly count of recorded opioid-involved synthetic events" if synthetic_used else "[Applied: document actual measure]",
            "population_denominator_available":False if synthetic_used else "[verify]",
            "fatal_outcome_available":False if synthetic_used else "[verify]",
            "provisional_definition":"month has one or more flagged source records" if synthetic_used else "[verify]",
            "figure_file":chart_path.name, "brief_file":brief_path.name}
(OUT / "metadata.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
(OUT / "README.md").write_text(
    f"# WISER Module 5 outputs\n\nPathway: {PATHWAY}\nSource: {source_name}\n\n"
    "Files: primary chart (PNG), one-page brief (PDF), monthly_recorded_events.csv, "
    "cde_alignment_table.csv, brief_text.md, chart_alt_text.txt, reflection.txt, metadata.json.\n\n"
    "For the supplied synthetic file, the measure is recorded opioid-involved event count, "
    "not a mortality count or a population rate. Provisional months are flagged from source rows. "
    "The CDE table contains topic-level candidates until target IDs and definitions are verified.\n",
    encoding="utf-8")
archive = shutil.make_archive("WISER_Module_5_outputs", "zip", root_dir=".", base_dir=str(OUT))
print("PDF:", brief_path, "| ZIP:", archive)
print("Files:", sorted(x.name for x in OUT.iterdir()))
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Outside Colab: download the ZIP from the file browser.")